# Chapter 6: Sentiment Analysis

Sentiment analysis (also called **opinion mining**) is a subfield of NLP that determines the sentiment or emotion expressed in a piece of text. The primary goal is to classify sentiment as **positive**, **negative**, or **neutral**.

**Common applications:**
- Social media monitoring (brand perception, public reaction)
- Customer feedback analysis (reviews, surveys, support tickets)
- Market research (consumer preferences, trend identification)
- Financial market analysis (investor sentiment from news/reports)
- Healthcare (patient feedback, public health trends)

In this notebook we explore three families of approaches:
1. **Rule-based** -- lexicon lookups and hand-crafted rules (TextBlob, AFINN, VADER)
2. **Machine learning** -- feature extraction + classical classifiers (Logistic Regression, Naive Bayes, SVM)
3. **Deep learning** -- neural architectures (CNN, LSTM, BERT)

## Setup

Install the required packages (run once).

In [ ]:
# Core packages
!pip install -q textblob afinn vaderSentiment scikit-learn numpy pandas matplotlib

# Deep learning (uncomment if needed -- these are large downloads)
# !pip install -q tensorflow transformers

In [ ]:
# Download TextBlob corpora (needed for the first run)
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

---
# 6.1 Rule-Based Approaches

Rule-based sentiment analysis relies on **manually crafted rules** and **sentiment lexicons** (word lists annotated with polarity scores). The general pipeline is:

1. **Tokenization** -- split text into words/tokens
2. **Normalization** -- lowercase, strip punctuation
3. **Lexicon lookup** -- assign each token a sentiment score from a lexicon (e.g. AFINN, SentiWordNet, NRC Emotion Lexicon)
4. **Rule application** -- aggregate token scores to produce an overall sentiment label

**Advantages:** transparent, easy to implement, domain-customizable.  
**Limitations:** limited coverage, no understanding of context/sarcasm, requires ongoing maintenance.

### 6.1.1 Sentiment Analysis with TextBlob

TextBlob provides a simple API that returns:
- **Polarity**: -1 (very negative) to +1 (very positive)
- **Subjectivity**: 0 (objective) to 1 (subjective)

In [ ]:
from textblob import TextBlob

# Sample text
text = "I love this product! It works wonderfully and the quality is excellent."

# Perform sentiment analysis
blob = TextBlob(text)
sentiment = blob.sentiment

print("Sentiment Analysis:")
print(f"  Polarity:     {sentiment.polarity}")
print(f"  Subjectivity: {sentiment.subjectivity}")

In [ ]:
# Analyze multiple sentences at once
sentences = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "The weather is okay, nothing special.",
    "I am so excited about the new movie release.",
]

for s in sentences:
    b = TextBlob(s)
    label = "Positive" if b.sentiment.polarity > 0 else ("Negative" if b.sentiment.polarity < 0 else "Neutral")
    print(f"[{label:>8}]  pol={b.sentiment.polarity:+.2f}  subj={b.sentiment.subjectivity:.2f}  | {s}")

### 6.1.2 Custom Rule-Based Analysis with AFINN

The **AFINN lexicon** assigns integer scores from -5 (very negative) to +5 (very positive) to individual words. We sum them for the full text.

In [ ]:
from afinn import Afinn

afinn = Afinn()

text = "I hate the traffic in this city. It makes commuting a nightmare."

sentiment_score = afinn.score(text)

if sentiment_score > 0:
    sentiment = "Positive"
elif sentiment_score < 0:
    sentiment = "Negative"
else:
    sentiment = "Neutral"

print("Sentiment Analysis:")
print(f"  Text:  {text}")
print(f"  Score: {sentiment_score}")
print(f"  Label: {sentiment}")

### 6.1.3 VADER -- Valence Aware Dictionary and sEntiment Reasoner

VADER is specifically tuned for **social media text**. It handles slang, emoticons, capitalization emphasis, and degree modifiers. It returns a `compound` score from -1 to +1 plus `pos`, `neu`, `neg` proportions.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

texts = [
    "I love this product! It's AMAZING!!! :)",
    "This is the worst service I have ever experienced.",
    "The movie was okay, nothing great.",
    "OMG this is SO good!!!",
    "I just love waiting in long lines... (sarcasm is hard for lexicon models)",
]

for t in texts:
    scores = analyzer.polarity_scores(t)
    label = "Positive" if scores['compound'] >= 0.05 else ("Negative" if scores['compound'] <= -0.05 else "Neutral")
    print(f"[{label:>8}]  compound={scores['compound']:+.4f}  | {t}")

### 6.1.4 Comparing Rule-Based Tools

Let's run all three analyzers on the same set of sentences so we can compare their outputs side by side.

In [ ]:
import pandas as pd

comparison_texts = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "The weather is okay today.",
    "I hate waiting in long lines.",
    "The food at the restaurant was fantastic.",
]

rows = []
for t in comparison_texts:
    tb = TextBlob(t).sentiment.polarity
    af = afinn.score(t)
    va = analyzer.polarity_scores(t)['compound']
    rows.append({"Text": t, "TextBlob": round(tb, 3), "AFINN": af, "VADER": round(va, 3)})

df = pd.DataFrame(rows)
df

---
# 6.2 Machine Learning Approaches

Machine learning approaches **train models on labeled data** so they can learn patterns automatically. The typical pipeline:

1. **Data collection** -- gather labeled text (positive / negative / neutral)
2. **Preprocessing** -- tokenization, normalization, stop-word removal
3. **Feature extraction** -- convert text to numbers (Bag-of-Words, TF-IDF, word embeddings)
4. **Model training** -- fit a classifier (Logistic Regression, Naive Bayes, SVM, Random Forest)
5. **Evaluation** -- accuracy, precision, recall, F1
6. **Prediction** -- classify new, unseen text

**Advantages:** better coverage, scalable, flexible across domains.  
**Limitations:** requires labeled data, more complex to build and tune, less interpretable than rules.

### 6.2.1 Feature Extraction with TF-IDF

**TF-IDF** (Term Frequency -- Inverse Document Frequency) weights words by how important they are within a document relative to the entire corpus. Common words get down-weighted; rare, informative words get up-weighted.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "I am very happy with my purchase.",
    "I am disappointed with the quality of this item.",
]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("\nTF-IDF matrix shape:", X.shape)
print("\nDense representation:")
pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out()).round(2)

### 6.2.2 Training a Logistic Regression Model

Logistic Regression is a simple yet effective baseline for binary text classification.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Larger corpus for a more meaningful demo
corpus = [
    "I love this product! It's amazing.",
    "Absolutely wonderful experience.",
    "Best purchase I have ever made.",
    "I am very happy with my purchase.",
    "Great quality and fast shipping.",
    "Highly recommend this to everyone.",
    "This is the worst service I have ever experienced.",
    "I am disappointed with the quality of this item.",
    "Terrible product, broke after one day.",
    "Awful experience, would not buy again.",
    "Poor quality and very slow delivery.",
    "Complete waste of money.",
]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]  # 1=positive, 0=negative

# Feature extraction
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42
)

# Train
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

# Evaluate
y_pred = model_lr.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

### 6.2.3 Naive Bayes Classifier

Multinomial Naive Bayes is a probabilistic model based on Bayes' theorem. It assumes feature independence and works surprisingly well for text.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

y_pred_nb = model_nb.predict(X_test)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb, target_names=["Negative", "Positive"]))

### 6.2.4 Support Vector Machine (SVM)

SVMs find the optimal hyperplane separating classes. They are especially effective in high-dimensional feature spaces like TF-IDF vectors.

In [ ]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC(max_iter=5000)
model_svm.fit(X_train, y_train)

y_pred_svm = model_svm.predict(X_test)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=["Negative", "Positive"]))

### 6.2.5 Evaluation Metrics Deep Dive

| Metric | What it measures |
|---|---|
| **Accuracy** | Fraction of all predictions that are correct |
| **Precision** | Of predicted positives, how many are truly positive? |
| **Recall** | Of actual positives, how many did we find? |
| **F1 Score** | Harmonic mean of precision and recall -- balances both |

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Using the Logistic Regression predictions as an example
print("Detailed metrics for Logistic Regression:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.2f}")
print(f"  Precision: {precision_score(y_test, y_pred, zero_division=0):.2f}")
print(f"  Recall:    {recall_score(y_test, y_pred, zero_division=0):.2f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.2f}")

### 6.2.6 Predicting New Text

Once trained, we can use any of the models above to classify unseen sentences.

In [ ]:
new_texts = [
    "The product is excellent and I love it.",
    "Horrible customer support, never buying again.",
    "It was fine, nothing special.",
]

new_X = vectorizer.transform(new_texts)

for text, pred in zip(new_texts, model_lr.predict(new_X)):
    label = "Positive" if pred == 1 else "Negative"
    print(f"[{label:>8}]  {text}")

---
# 6.3 Deep Learning Approaches

Deep learning models use **neural networks** to learn complex, hierarchical representations from raw text. Key architectures:

| Architecture | Strengths |
|---|---|
| **CNN** | Captures local n-gram patterns efficiently |
| **RNN / LSTM** | Models sequential dependencies; LSTMs solve the vanishing gradient problem |
| **Transformer (BERT)** | Self-attention captures long-range context; state-of-the-art performance |

**Advantages:** state-of-the-art accuracy, automatic feature learning, handles complex/long text.  
**Limitations:** computationally expensive, needs large labeled datasets, "black box" interpretability.

> **Note:** The cells below require `tensorflow` and `transformers`. Uncomment the pip install cell at the top if you have not installed them yet.

### 6.3.1 Sentiment Analysis with a CNN

A 1-D convolutional network slides filters over the embedding sequence to detect local patterns (like n-grams), then uses global max-pooling to aggregate them.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, GlobalMaxPooling1D, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Sample corpus and labels
corpus = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "I am very happy with my purchase.",
    "I am disappointed with the quality of this item.",
]
labels = [1, 0, 1, 0]

# Tokenize and pad
tokenizer_dl = Tokenizer(num_words=5000)
tokenizer_dl.fit_on_texts(corpus)
X = tokenizer_dl.texts_to_sequences(corpus)
X = pad_sequences(X, maxlen=10)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42
)

# Build CNN
model_cnn = Sequential([
    Embedding(input_dim=5000, output_dim=50, input_length=10),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(10, activation='relu'),
    Dense(1, activation='sigmoid'),
])

model_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_cnn.summary()

In [ ]:
# Train the CNN
model_cnn.fit(
    np.array(X_train), np.array(y_train),
    epochs=5, verbose=1,
    validation_data=(np.array(X_test), np.array(y_test)),
)

loss, accuracy = model_cnn.evaluate(np.array(X_test), np.array(y_test))
print(f"\nCNN Test Accuracy: {accuracy:.2f}")

In [ ]:
# Predict on new text
new_text = ["The product is excellent and I love it."]
new_seq = tokenizer_dl.texts_to_sequences(new_text)
new_padded = pad_sequences(new_seq, maxlen=10)
prediction = model_cnn.predict(new_padded)
print(f"Prediction: {'Positive' if prediction[0][0] > 0.5 else 'Negative'} (score={prediction[0][0]:.4f})")

### 6.3.2 Sentiment Analysis with an LSTM

LSTMs (Long Short-Term Memory networks) are a specialized RNN that use **gates** (input, forget, output) to control information flow, solving the vanishing gradient problem and capturing long-range dependencies.

In [ ]:
from tensorflow.keras.layers import LSTM

# Reuse the same tokenized data from above
model_lstm = Sequential([
    Embedding(input_dim=5000, output_dim=50, input_length=10),
    LSTM(100),
    Dense(1, activation='sigmoid'),
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_lstm.summary()

In [ ]:
# Train the LSTM
model_lstm.fit(
    np.array(X_train), np.array(y_train),
    epochs=5, verbose=1,
    validation_data=(np.array(X_test), np.array(y_test)),
)

loss, accuracy = model_lstm.evaluate(np.array(X_test), np.array(y_test))
print(f"\nLSTM Test Accuracy: {accuracy:.2f}")

In [ ]:
# Predict with LSTM
prediction = model_lstm.predict(new_padded)
print(f"Prediction: {'Positive' if prediction[0][0] > 0.5 else 'Negative'} (score={prediction[0][0]:.4f})")

### 6.3.3 Transformer-Based Models -- BERT

BERT (Bidirectional Encoder Representations from Transformers) uses **self-attention** to capture context from both directions simultaneously. Pre-trained on massive corpora, it can be fine-tuned for downstream tasks like sentiment analysis with relatively small labeled datasets.

> **Note:** This cell downloads the `bert-base-uncased` model (~440 MB). Ensure you have sufficient disk space and bandwidth.

In [ ]:
import numpy as np
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split

# Sample corpus and labels
corpus = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "I am very happy with my purchase.",
    "I am disappointed with the quality of this item.",
]
labels = [1, 0, 1, 0]

# Tokenize with BERT tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
X_bert = bert_tokenizer(
    corpus, padding=True, truncation=True, max_length=10, return_tensors='tf'
)

# Train/test split
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_bert['input_ids'], labels, test_size=0.25, random_state=42
)

# Load pre-trained BERT for classification
model_bert = TFBertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)

model_bert.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

# Fine-tune
model_bert.fit(
    X_train_b, np.array(y_train_b),
    epochs=3, batch_size=8,
    validation_data=(X_test_b, np.array(y_test_b)),
)

# Evaluate
loss, accuracy = model_bert.evaluate(X_test_b, np.array(y_test_b))
print(f"\nBERT Test Accuracy: {accuracy:.2f}")

In [ ]:
# Predict sentiment of new text with BERT
new_text = ["The product is excellent and I love it."]
new_enc = bert_tokenizer(
    new_text, padding=True, truncation=True, max_length=10, return_tensors='tf'
)
prediction = model_bert.predict(new_enc['input_ids'])
predicted_label = np.argmax(prediction.logits, axis=1)[0]
print(f"Prediction: {'Positive' if predicted_label == 1 else 'Negative'}")

### 6.3.4 Using a Pre-trained Sentiment Pipeline (Quick Alternative)

If you just need predictions without fine-tuning, Hugging Face provides a convenient `pipeline` API with models already trained on sentiment data.

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis")

examples = [
    "I love this product! It's amazing.",
    "This is the worst service I have ever experienced.",
    "The weather is okay, nothing special.",
]

results = sentiment_pipeline(examples)
for text, result in zip(examples, results):
    print(f"[{result['label']:>8}  {result['score']:.4f}]  {text}")

---
# 6.4 Approach Comparison Summary

| | Rule-Based | Machine Learning | Deep Learning |
|---|---|---|---|
| **Interpretability** | High | Medium | Low ("black box") |
| **Setup complexity** | Low | Medium | High |
| **Data requirements** | None (lexicon only) | Moderate labeled data | Large labeled data |
| **Handles context** | Limited | Somewhat | Very well |
| **Handles sarcasm** | Poorly | Depends on data | Better, but still hard |
| **Compute cost** | Minimal | Low-moderate | High (GPU/TPU) |
| **Accuracy ceiling** | Lower | Good | State-of-the-art |

---
# Practical Exercises

Try these on your own before looking at the solutions below.

### Exercise 1: Rule-Based Sentiment Analysis with TextBlob

**Task:** Perform sentiment analysis on the following sentences using TextBlob:
- "The weather is terrible today."
- "I am so excited about the new movie release."

In [ ]:
# Exercise 1 -- your code here


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from textblob import TextBlob

texts = [
    "The weather is terrible today.",
    "I am so excited about the new movie release.",
]

for text in texts:
    blob = TextBlob(text)
    sentiment = blob.sentiment
    print(f"Text: {text}")
    print(f"  Polarity: {sentiment.polarity}, Subjectivity: {sentiment.subjectivity}")
    print()
```

Expected output:
- "terrible" -> Polarity: -1.0, Subjectivity: 1.0
- "excited" -> Polarity: 0.8, Subjectivity: 1.0
</details>

### Exercise 2: Custom Rule-Based Sentiment Analysis with AFINN

**Task:** Use the AFINN lexicon to analyze:
- "I hate waiting in long lines."
- "The food at the restaurant was fantastic."

In [ ]:
# Exercise 2 -- your code here


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from afinn import Afinn

afinn = Afinn()

texts = [
    "I hate waiting in long lines.",
    "The food at the restaurant was fantastic.",
]

for text in texts:
    score = afinn.score(text)
    label = "Positive" if score > 0 else "Negative" if score < 0 else "Neutral"
    print(f"Text: {text}")
    print(f"  Score: {score}  ->  {label}")
    print()
```

Expected: "hate" -> Score: -3.0 (Negative); "fantastic" -> Score: 4.0 (Positive)
</details>

### Exercise 3: Sentiment Analysis with Logistic Regression

**Task:** Train a Logistic Regression model on these sentences and evaluate it:
- Sentences: `["I love this product!", "This is the worst service.", "I am happy with my purchase.", "The quality is terrible."]`
- Labels: `[1, 0, 1, 0]`

In [ ]:
# Exercise 3 -- your code here


<details>
<summary><b>Click to reveal solution</b></summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

corpus = [
    "I love this product!",
    "This is the worst service.",
    "I am happy with my purchase.",
    "The quality is terrible.",
]
labels = [1, 0, 1, 0]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
```
</details>

### Exercise 4: Sentiment Analysis with an LSTM

**Task:** Build and train an LSTM model using the same sentences as Exercise 3. Predict the sentiment of `"The product is excellent and I love it."`

In [ ]:
# Exercise 4 -- your code here


<details>
<summary><b>Click to reveal solution</b></summary>

```python
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

corpus = [
    "I love this product!",
    "This is the worst service.",
    "I am happy with my purchase.",
    "The quality is terrible.",
]
labels = [1, 0, 1, 0]

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(corpus)
X = tokenizer.texts_to_sequences(corpus)
X = pad_sequences(X, maxlen=10)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.25, random_state=42
)

model = Sequential([
    Embedding(input_dim=5000, output_dim=50, input_length=10),
    LSTM(100),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(np.array(X_train), np.array(y_train), epochs=5, verbose=1,
          validation_data=(np.array(X_test), np.array(y_test)))

loss, accuracy = model.evaluate(np.array(X_test), np.array(y_test))
print(f"Accuracy: {accuracy:.2f}")

new_text = ["The product is excellent and I love it."]
new_seq = tokenizer.texts_to_sequences(new_text)
new_padded = pad_sequences(new_seq, maxlen=10)
pred = model.predict(new_padded)
print(f"Prediction: {'Positive' if pred[0][0] > 0.5 else 'Negative'}")
```
</details>

### Exercise 5: Compare All Rule-Based Analyzers on Your Own Text

**Task:** Pick 5 sentences of your own (mix of positive, negative, neutral) and run TextBlob, AFINN, and VADER on each. Create a DataFrame showing the results. Which analyzer do you find most accurate for your sentences?

In [ ]:
# Exercise 5 -- your code here
